In [1]:
!pip install pdfplumber pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.8 MB/s eta 0:00:00


In [20]:
from google.colab import files
uploaded = files.upload()

zip_file = list(uploaded.keys())[0]
zip_file


Saving credit_card_statements.zip to credit_card_statements (1).zip


'credit_card_statements (1).zip'

In [21]:
import zipfile, os

extract_path = "/content/statements"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as z:
    z.extractall(extract_path)

# if nested folder exists, auto-detect
for root, dirs, files in os.walk(extract_path):
    if any(f.endswith(".pdf") for f in files):
        pdf_folder = root
        break

pdf_folder


'/content/statements/credit_card_statements'

In [22]:
import pdfplumber

def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text


In [23]:
import re

def parse_fields(text):

    data = {
        "issuer": "Unknown",
        "last4": None,
        "billing_period": None,
        "due_date": None,
        "total_due": None,
        "closing_balance": None
    }

    t = text.lower()

    # -----------------------
    # IDENTIFY ISSUER
    # -----------------------
    if "axis" in t: data["issuer"] = "Axis Bank"
    elif "hdfc" in t: data["issuer"] = "HDFC Bank"
    elif "icici" in t: data["issuer"] = "ICICI Bank"
    elif "sbi" in t: data["issuer"] = "SBI"
    elif "citi" in t: data["issuer"] = "Citi"


    # -----------------------
    # CARD LAST 4 DIGITS
    # -----------------------
    patterns_last4 = [
        r"XXXX\s+XXXX\s+XXXX\s+(\d{4})",
        r"xxxx\s+xxxx\s+xxxx\s+(\d{4})",
        r"ending\s+(\d{4})",
        r"(\d{4})$",
    ]
    for p in patterns_last4:
        m = re.search(p, text, re.I)
        if m:
            data["last4"] = m.group(1)
            break


    # -----------------------
    # BILLING/STATEMENT PERIOD
    # supports: 01 Apr – 30 Apr 2025
    # -----------------------
    patterns_period = [
        r"Statement\s*Period[:\-]?\s*(.*?\d{4})",
        r"Billing\s*Period[:\-]?\s*(.*?\d{4})",
        r"Period[:\-]?\s*(\d{1,2}.*?\d{4})"
    ]
    for p in patterns_period:
        m = re.search(p, text, flags=re.I)
        if m:
            data["billing_period"] = m.group(1).strip()
            break


    # -----------------------
    # PAYMENT DUE DATE
    # -----------------------
    patterns_due = [
        r"Payment\s*Due\s*Date[:\-]?\s*([0-9]{1,2}\s*\w+\s*[0-9]{4})",
        r"Due\s*Date[:\-]?\s*([0-9]{1,2}\s*\w+\s*[0-9]{4})"
    ]
    for p in patterns_due:
        m = re.search(p, text, flags=re.I)
        if m:
            data["due_date"] = m.group(1)
            break


    # -----------------------
    # TOTAL DUE
    # -----------------------
    patterns_total = [
        r"Total\s*Amount\s*Due[:\-]?\s*₹?([\d,]+\.\d{2})",
        r"Amount\s*Due[:\-]?\s*₹?([\d,]+\.\d{2})"
    ]
    for p in patterns_total:
        m = re.search(p, text, flags=re.I)
        if m:
            data["total_due"] = m.group(1)
            break


    # -----------------------
    # CLOSING BALANCE (if available)
    # -----------------------
    patterns_closing = [
        r"Closing\s*Balance[:\-]?\s*₹?([\d,]+\.\d{2})",
        r"Closing\s*Amount[:\-]?\s*₹?([\d,]+\.\d{2})"
    ]
    for p in patterns_closing:
        m = re.search(p, text, flags=re.I)
        if m:
            data["closing_balance"] = m.group(1)
            break

    return data


In [24]:
import os

def parse_all(folder):
    results = []

    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            path = f"{folder}/{file}"
            text = extract_text(path)
            fields = parse_fields(text)
            fields["file"] = file
            results.append(fields)

    return results


output = parse_all(pdf_folder)
output


[{'issuer': 'HDFC Bank',
  'last4': '4821',
  'billing_period': '01 Jan 2025',
  'due_date': '15 Feb 2025',
  'total_due': '18,540.00',
  'closing_balance': None,
  'file': 'hdfc.1.pdf'},
 {'issuer': 'Citi',
  'last4': '6678',
  'billing_period': '10 Jan – 09 Feb 2025',
  'due_date': '25 Feb 2025',
  'total_due': '32,780.00',
  'closing_balance': None,
  'file': 'citi.1.pdf'},
 {'issuer': 'Citi',
  'last4': '9011',
  'billing_period': '01 Feb – 28 Feb 2025',
  'due_date': '18 Mar 2025',
  'total_due': '17,990.00',
  'closing_balance': None,
  'file': 'citi.2.pdf'},
 {'issuer': 'ICICI Bank',
  'last4': '6710',
  'billing_period': '10 Mar – 09 Apr 2025',
  'due_date': '24 Apr 2025',
  'total_due': '17,560.00',
  'closing_balance': None,
  'file': 'icici.3.pdf'},
 {'issuer': 'Citi',
  'last4': '4472',
  'billing_period': '01 Mar – 31 Mar 2025',
  'due_date': '20 Apr 2025',
  'total_due': '23,500.00',
  'closing_balance': None,
  'file': 'citi.3.pdf'},
 {'issuer': 'SBI',
  'last4': '1142',

In [25]:
import pandas as pd
df = pd.DataFrame(output)
df


,issuer,last4,billing_period,due_date,total_due,closing_balance,file
0,HDFC Bank,4821,01 Jan 2025,15 Feb 2025,"18,540.00",None,hdfc.1.pdf
1,Citi,6678,10 Jan – 09 Feb 2025,25 Feb 2025,"32,780.00",None,citi.1.pdf
2,Citi,9011,01 Feb – 28 Feb 2025,18 Mar 2025,"17,990.00",None,citi.2.pdf
3,ICICI Bank,6710,10 Mar – 09 Apr 2025,24 Apr 2025,"17,560.00",None,icici.3.pdf
4,Citi,4472,01 Mar – 31 Mar 2025,20 Apr 2025,"23,500.00",None,citi.3.pdf
5,SBI,1142,01 Jan – 31 Jan 2025,17 Feb 2025,"12,780.50",None,sbi.1.pdf
6,Axis Bank,6643,01 Apr – 30 Apr 2025,10 May 2025,"25,890.00",None,axis.3.pdf
7,Axis Bank,7229,01 Feb – 28 Feb 2025,12 Mar 2025,"9,640.00",None,axis.1.pdf
8,HDFC Bank,8891,01 Mar 2025,15 Apr 2025,"10,340.00",None,hdfc.3.pdf
9,HDFC Bank,1934,01 Feb 2025,14 Mar 2025,"22,940.00",None,hdfc.2.pdf
